# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIRˆ<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant JSON-LD schema:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Install mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIRˆ<sup>2</sup> dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Access Croissant metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, referencing entities by their `@id`.

Let's examine which record sets are present and identify the fields within each record set.

In [ ]:
# List all record sets by @id
record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in getattr(metadata, 'recordSet', [])]
if not record_sets:
    print("No record sets were listed in the top-level metadata. Attempting to infer from mlcroissant...")
    # mlcroissant automatically discovers record sets
    # List their @id
    discovered_record_sets = dataset.list_record_sets()
    record_sets = [rs['@id'] for rs in discovered_record_sets]
    for rs in discovered_record_sets:
        print(f"Record Set @id: {rs['@id']} Name: {rs.get('name','(no name)')}")
else:
    for rs_id in record_sets:
        print(f"Record Set @id: {rs_id}")

### Show fields within each record set

We can now iterate through each record set and show the available fields with their `@id`.

In [ ]:
for rs_id in record_sets:
    print(f"\nFields in record set {rs_id}:")
    record_set_metadata = dataset.get_record_set_metadata(rs_id)
    fields = record_set_metadata.get('fields', [])
    for field in fields:
        field_id = field.get('@id', '(unknown)')
        field_name = field.get('name', '(no name)')
        print(f"Field @id: {field_id} | Name: {field_name}")

## 3. Data Extraction
Load data from a selected record set into a pandas DataFrame for analysis.
Entities are referenced strictly by their `@id`.

We will extract all available record sets and examine their contents.

In [ ]:
# Extract all dataframes from each discovered record set
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Record Set {rs_id}: Columns -> {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"Record Set {rs_id} returned no records.")

## 4. Exploratory Data Analysis (EDA)
Apply processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping by categorical fields.

**Note:** Since the dataset is small (N=77), we'll demonstrate filtering, normalization, and grouping using available numeric and categorical fields.

In [ ]:
# Choose the main record set based on largest number of records
record_set_id = max(dataframes, key=lambda k: len(dataframes[k])) if dataframes else None
if record_set_id:
    df = dataframes[record_set_id]
    # List numeric fields (by dtype)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields: {numeric_fields}")
    if numeric_fields:
        # Filter by arbitrary threshold on first numeric field
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean()  # Use mean to filter above-average
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].__getitem__(slice(0,2))].head())
        # Try grouping by a non-numeric field
        group_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() < 10]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field}, showing mean of {numeric_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot histogram and boxplot for any available numeric field, and, if possible, visualize relationship with a categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_fields:
    numeric_field_id = numeric_fields[0]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    # Boxplot by group
    if group_fields:
        group_field = group_fields[0]
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field} in {record_set_id}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded FAIRˆ<sup>2</sup> dataset metadata and records via `mlcroissant`.
- Surveyed all available record sets and their fields by `@id`.
- Extracted primary tabular data and performed exploratory analyses using pandas.
- Demonstrated basic filtering, normalization, grouping, and visualization techniques.

This workflow provides a robust foundation for systematic FAIR dataset exploration and reproducible analysis using Croissant schemas.